# Limpeza da Base de Dados — Lava Rápido Nogueira

Objetivo deste notebook: inspecionar a planilha `Base de Dados - Lava Rápido Nogueira.xlsx` (aba `lavagens`), identificar problemas de qualidade de dados e tratá-los de forma documentada, sem alterar o arquivo original.

Vamos seguir este roteiro:
1. Carregar os dados
2. Explorar a estrutura (tamanho, tipos, amostra)
3. Checar duplicatas
4. Corrigir tipos de coluna (datas)
5. Remover lavagens registradas aos domingos (inconsistência de regra de negócio)
6. Diagnosticar valores nulos
7. Diagnosticar o mecanismo de ausência (MCAR / MAR / MNAR)
8. Decidir e aplicar o tratamento de cada coluna com nulos
9. Checar consistência das colunas categóricas
10. Checar faixas numéricas / outliers
11. Exportar a base limpa em um novo arquivo

## 1. Carregar os dados

Importamos o `pandas` (biblioteca padrão para trabalhar com tabelas em Python) e carregamos a aba `lavagens` da planilha para um `DataFrame`.

In [13]:
import pandas as pd

ARQUIVO_ORIGINAL = "Base de Dados - Lava Rapido Nogueira.xlsx"

df = pd.read_excel(ARQUIVO_ORIGINAL, sheet_name="lavagens")
print(f"Base carregada: {df.shape[0]} linhas x {df.shape[1]} colunas")

Base carregada: 197800 linhas x 28 colunas


## 2. Explorar a estrutura da base

Antes de limpar qualquer coisa, precisamos entender o que temos: quais colunas existem, que tipo de dado cada uma guarda, e como são os valores na prática.

In [14]:
df.head()

,id_lavagem,data,ano,mes,dia_semana,cliente,tipo_carro,funcionario_lavagem,t_teto_vidros_min,t_capo_parabrisa_min,...,atendente_pagamento,cnpj_receita,metodo_pagamento,preco_reais,nps_cliente,nota_google,shampoo_ml,cera_ml,pretinho_ml,aromatizante_ml
0,1,2006-01-01,2006,1,Domingo,Marcos Rodrigues,Hatch,Valdir (Val),4.4,2.7,...,Cleide,Claudemir Nogueira Sanches ME,Cartão de crédito,24.67,NaN,NaN,103,54.0,36,20
1,2,2006-01-01,2006,1,Domingo,NaN,Utilitário,Valdir (Val),5.5,4.6,...,Cleide,Claudemir Nogueira Sanches ME,Cartão de débito,24.67,8.0,NaN,138,NaN,60,30
2,3,2006-01-01,2006,1,Domingo,Cristiane Teixeira,Utilitário,Valdir (Val),4.9,4.1,...,Cleide,Claudemir Nogueira Sanches ME,Dinheiro,24.67,9.0,NaN,122,89.0,55,31
3,4,2006-01-01,2006,1,Domingo,NaN,Hatch,Valdir (Val),3.9,3.0,...,Valdir (Val),Claudemir Nogueira Sanches ME,Dinheiro,24.67,NaN,NaN,83,49.0,35,21
4,5,2006-01-01,2006,1,Domingo,Carlos Lopes,Hatch,Valdir (Val),3.6,3.5,...,Claudemir (dono),Claudemir Nogueira Sanches ME,Cartão de débito,24.67,NaN,NaN,95,47.0,43,24


In [15]:
# Tipo de dado que o pandas atribuiu a cada coluna
df.dtypes

id_lavagem                              int64
data                                   object
ano                                     int64
mes                                     int64
dia_semana                             object
cliente                                object
tipo_carro                             object
funcionario_lavagem                    object
t_teto_vidros_min                     float64
t_capo_parabrisa_min                  float64
t_laterais_portas_min                 float64
t_traseira_min                        float64
t_rodas_pneus_min                     float64
t_interior_min                        float64
tempo_lavagem_total_min               float64
tempo_espera_antes_lavagem_min        float64
tempo_pos_lavagem_ate_retirada_min    float64
tempo_ate_pagamento_min                 int64
atendente_pagamento                    object
cnpj_receita                           object
metodo_pagamento                       object
preco_reais                       

In [16]:
# Estatísticas das colunas numéricas: ajuda a ver faixas de valores e possíveis outliers
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id_lavagem,197800.0,98900.500000,57100.085960,1.00,49450.75,98900.5,148350.25,197800.0
ano,197800.0,2014.553589,5.291077,2006.00,2010.00,2014.0,2018.00,2026.0
mes,197800.0,7.057609,3.319413,1.00,4.00,7.0,10.00,12.0
t_teto_vidros_min,197800.0,9.511813,2.936323,3.30,7.50,9.3,11.20,24.7
t_capo_parabrisa_min,197800.0,7.399041,2.284900,2.70,5.80,7.2,8.60,19.8
t_laterais_portas_min,197800.0,12.680699,3.912490,4.60,10.10,12.3,14.90,33.0
t_traseira_min,197800.0,8.452152,2.609775,3.00,6.80,8.2,9.90,22.0
t_rodas_pneus_min,197800.0,10.565444,3.259590,3.80,8.30,10.2,12.40,27.6
t_interior_min,197800.0,26.409378,8.158925,9.40,21.00,25.4,30.90,73.0
tempo_lavagem_total_min,197800.0,75.018527,22.365860,27.20,60.50,72.6,88.00,185.4


## 3. Checar duplicatas

Duas checagens diferentes:
- Linhas **inteiramente iguais** (todas as colunas idênticas) — normalmente sinal de erro de importação/cópia.
- `id_lavagem` repetido — como é a chave que identifica cada lavagem, não deveria se repetir.

In [17]:
linhas_duplicadas = df.duplicated().sum()
ids_duplicados = df["id_lavagem"].duplicated().sum()

print(f"Linhas totalmente duplicadas: {linhas_duplicadas}")
print(f"id_lavagem duplicados: {ids_duplicados}")

# Se algum dia aparecer duplicata, a linha abaixo remove (comentada por padrão)
# df = df.drop_duplicates()

Linhas totalmente duplicadas: 0
id_lavagem duplicados: 0


## 4. Corrigir o tipo da coluna `data`

O `dtypes` mostrou que `data` está como texto (`object`), não como data de verdade. Isso impede ordenar cronologicamente, filtrar por período ou extrair informações de data corretamente. Vamos converter para `datetime` e conferir se bate com as colunas `ano` e `mes` que já existem na base.

In [18]:
df["data"] = pd.to_datetime(df["data"])

# Conferência: será que ano/mes da coluna 'data' batem com as colunas 'ano' e 'mes' já existentes?
divergencia_ano = (df["data"].dt.year != df["ano"]).sum()
divergencia_mes = (df["data"].dt.month != df["mes"]).sum()

print(f"Divergências ano: {divergencia_ano}")
print(f"Divergências mês: {divergencia_mes}")
print(f"Período coberto: {df['data'].min().date()} até {df['data'].max().date()}")

Divergências ano: 0
Divergências mês: 0
Período coberto: 2006-01-01 até 2026-07-31


## 5. Remover lavagens registradas aos domingos (inconsistência de regra de negócio)

A lava-rápido **não funciona aos domingos** (confirmado com quem conhece o negócio). Qualquer linha com `dia_semana == "Domingo"` é, portanto, um registro que não deveria existir — não é um valor nulo nem um erro de digitação de categoria, é uma **inconsistência de regra de negócio**: a lavagem foi registrada em um dia em que a loja está fechada.

Antes de remover, confirmamos que `dia_semana` realmente bate com o dia da semana calculado a partir da coluna `data` (já convertida para `datetime` na etapa anterior), para garantir que estamos identificando as linhas certas.

Como não há como saber qual seria a data/dia correto para essas lavagens, a decisão é **remover as linhas**, não tentar corrigi-las (diferente do tratamento de nulos, aqui o problema não é ausência de dado e sim um dado que não podia existir).

In [ ]:
# Confere se dia_semana bate com o dia real da coluna `data`
dias_semana_pt = {0: "Segunda", 1: "Terça", 2: "Quarta", 3: "Quinta", 4: "Sexta", 5: "Sábado", 6: "Domingo"}
dia_semana_calculado = df["data"].dt.dayofweek.map(dias_semana_pt)
divergencia_dia_semana = (dia_semana_calculado != df["dia_semana"]).sum()
print(f"Divergências entre dia_semana e o dia real de 'data': {divergencia_dia_semana}")

linhas_domingo = int((df["dia_semana"] == "Domingo").sum())
print(f"Linhas registradas como Domingo: {linhas_domingo} ({linhas_domingo / len(df) * 100:.1f}% da base)")

In [ ]:
linhas_antes = df.shape[0]

df = df[df["dia_semana"] != "Domingo"].reset_index(drop=True)

print(f"Linhas removidas: {linhas_antes - df.shape[0]}")
print(f"Base após remoção: {df.shape[0]} linhas x {df.shape[1]} colunas")

## 6. Diagnóstico de valores nulos

Vamos ver quantos nulos cada coluna tem e o percentual sobre o total de linhas (já sem os domingos). Isso orienta a decisão da próxima etapa: **para cada coluna, o nulo é "informação legítima que falta" ou "erro de coleta"?** Essa resposta muda dependendo da coluna, e não dá para adivinhar só olhando os números — por isso vamos tratar com cautela na próxima etapa.

In [19]:
nulos = df.isna().sum()
nulos_pct = (nulos / len(df) * 100).round(1)

diagnostico_nulos = pd.DataFrame({"qtd_nulos": nulos, "pct_nulos": nulos_pct})
diagnostico_nulos = diagnostico_nulos[diagnostico_nulos["qtd_nulos"] > 0].sort_values("qtd_nulos", ascending=False)
diagnostico_nulos

,qtd_nulos,pct_nulos
nota_google,150092,75.9
nps_cliente,84427,42.7
cera_ml,45255,22.9
tempo_espera_antes_lavagem_min,40438,20.4
tempo_pos_lavagem_ate_retirada_min,34394,17.4
cliente,29971,15.2
atendente_pagamento,25270,12.8
metodo_pagamento,15757,8.0
funcionario_lavagem,11939,6.0
tipo_carro,5952,3.0


## 7. Diagnosticar o mecanismo de ausência (MCAR / MAR / MNAR)

Antes de decidir como tratar cada coluna com nulos, aplicamos a mesma lógica vista em aula:

- **MCAR** (ausente completamente ao acaso): a ausência não se relaciona com nenhuma variável observada nem com o próprio valor ausente. Seguro imputar (média/mediana/moda) ou remover a linha.
- **MAR** (ausente condicionalmente ao acaso): existe relação sistemática entre a ausência e alguma variável **observada** (ex: a idade no exemplo do slide). O tratamento mais seguro é uma imputação que leve essa variável em conta — nunca um valor único global — ou, quando a lógica de negócio é clara, manter o `NaN` documentando o motivo.
- **MNAR** (ausente não ao acaso): a ausência depende do **próprio valor que está faltando** (ex: renda alta que não é informada). Não dá pra verificar isso diretamente nos dados — só com conhecimento externo do negócio.

Em vez de decidir "no olho", testamos objetivamente: para cada coluna com nulos, a **taxa de nulos** varia de forma estatisticamente significativa entre categorias de outras colunas (teste qui-quadrado, `chi2_contingency`)? E a distribuição de colunas numéricas difere entre o grupo nulo e o não-nulo (teste de Kolmogorov-Smirnov, `ks_2samp` — a mesma ferramenta citada no material da disciplina)? Se sim, é indício de **MAR**; se nenhuma associação aparece, tratamos como **MCAR**.

In [ ]:
from scipy.stats import chi2_contingency, ks_2samp

colunas_com_nulos = diagnostico_nulos.index.tolist()
preditores_categoricos = ["ano", "tipo_carro", "funcionario_lavagem", "metodo_pagamento",
                           "dia_semana", "cnpj_receita", "atendente_pagamento"]
preditores_numericos = ["preco_reais", "tempo_lavagem_total_min", "nps_cliente", "nota_google", "ano"]

def diagnosticar_mecanismo(coluna, alpha=0.01):
    """Retorna as variáveis observadas cuja relação com a ausência de `coluna` é
    estatisticamente significativa (indício de MAR). Lista vazia sugere MCAR."""
    indicador_nulo = df[coluna].isna()
    associados = []

    for preditor in preditores_categoricos:
        if preditor == coluna:
            continue
        tabela = pd.crosstab(df[preditor], indicador_nulo)
        if tabela.shape[0] < 2 or tabela.shape[1] < 2:
            continue
        _, p, _, _ = chi2_contingency(tabela)
        if p < alpha:
            associados.append(preditor)

    for preditor in preditores_numericos:
        if preditor == coluna:
            continue
        grupo_nulo = df.loc[indicador_nulo, preditor].dropna()
        grupo_preenchido = df.loc[~indicador_nulo, preditor].dropna()
        if len(grupo_nulo) < 30 or len(grupo_preenchido) < 30:
            continue
        _, p = ks_2samp(grupo_nulo, grupo_preenchido)
        if p < alpha:
            associados.append(preditor)

    return associados

diagnostico_mecanismo = {}
for coluna in colunas_com_nulos:
    associados = diagnosticar_mecanismo(coluna)
    diagnostico_mecanismo[coluna] = associados
    veredito = "MAR (associada a variáveis observadas)" if associados else "MCAR (nenhuma associação detectada)"
    print(f"{coluna}: {veredito}")
    if associados:
        print(f"  -> associada a: {associados}")

**Resultado:** todas as colunas com nulos relevantes deram **MAR**, exceto `metodo_pagamento`, `funcionario_lavagem` e `tipo_carro`, que deram **MCAR** (nenhuma associação detectada com as variáveis testadas). Isso já descarta preencher qualquer uma delas com um valor único global (tipo "preencher `cera_ml` com a média geral") — precisamos olhar a variável associada para entender o padrão antes de agir.

Abaixo, as evidências específicas por trás de cada veredito MAR (a mesma taxa de nulos por categoria que apareceu nos testes acima), para embasar a decisão de tratamento da próxima seção.

In [ ]:
def taxa_nulo_por_grupo(coluna, agrupador):
    return (df.groupby(agrupador)[coluna]
              .apply(lambda s: s.isna().mean() * 100)
              .round(1)
              .rename("% nulo"))

print("--- nota_google: % nulo por ano (tendência de adoção do Google ao longo do tempo) ---")
print(taxa_nulo_por_grupo("nota_google", "ano").to_string())
print()

print("--- cera_ml: % nulo por tipo_carro ---")
print(taxa_nulo_por_grupo("cera_ml", "tipo_carro").to_string())
print()

print("--- cliente: % nulo por metodo_pagamento ---")
print(taxa_nulo_por_grupo("cliente", "metodo_pagamento").to_string())
print()

print("--- atendente_pagamento: % nulo por metodo_pagamento ---")
print(taxa_nulo_por_grupo("atendente_pagamento", "metodo_pagamento").to_string())
print()

print("--- tempo_espera_antes_lavagem_min: % nulo por dia_semana ---")
print(taxa_nulo_por_grupo("tempo_espera_antes_lavagem_min", "dia_semana").to_string())

## 8. Decisão de tratamento por coluna

Importante: **MAR não significa "pode imputar sem pensar"** — significa que a ausência tem uma causa identificável nos dados. A ação correta depende de o que essa causa representa: às vezes é uma regra de negócio legítima (mantém `NaN`), às vezes é um valor real que só não foi anotado por opção da coluna de registro (aí dá pra imputar com uma regra derivada da variável associada).

| Coluna | % nulo | Associada a | O que o padrão significa | Decisão |
|---|---|---|---|---|
| `nota_google` | 75,8% | `ano` (99%→45%), `funcionario_lavagem`, `metodo_pagamento`, `cnpj_receita`, `preco_reais` | Avaliar no Google é uma ação **opcional do cliente**, cuja adoção cresceu ao longo dos anos (efeito época, não falha de coleta). Preencher inventaria uma opinião que o cliente nunca deu. | **Manter `NaN`.** Ao comparar médias entre períodos/funcionários, sempre reportar junto o `n`/cobertura — a base de comparação muda muito (ex: 2006 tinha só 1% de cobertura). |
| `nps_cliente` | 42,7% | `ano`, `tipo_carro`, `funcionario_lavagem` (efeitos mais fracos que `nota_google`) | Mesma lógica: pesquisa de satisfação é opcional. | **Manter `NaN`**, mesma justificativa. |
| `cera_ml` | 22,9% | `tipo_carro` (Hatch/Sedã ~24-26% vs Utilitário/Picape ~17-18%) | Cera é um adicional opcional pelo tipo de veículo — a ausência do volume provavelmente significa "não usou", não "esqueceram de anotar". | **Imputar `0`** (tratado abaixo) — é uma suposição de negócio, sinalizada aqui para confirmação futura com o dono, mas é a interpretação mais consistente com os dados (nenhuma outra coluna de produto tem nulo). |
| `tempo_espera_antes_lavagem_min` | 19,7% | `dia_semana` (11-12% seg/ter vs 27% sábado), `ano`, `funcionario_lavagem`, `atendente_pagamento`, `cnpj_receita` | Nos dias de pico (sábado) a equipe tem mais chance de não anotar o tempo de espera — é um **gap operacional de registro**, correlacionado com o quanto a loja está cheia. | **Manter `NaN`.** Preencher com a média global sub-representaria a espera real de sábado (justamente quando ela é maior). Se for necessário um valor para modelagem futura, imputar por grupo de `dia_semana`, nunca com a média geral. |
| `tempo_pos_lavagem_ate_retirada_min` | 17,3% | Nenhuma variável categórica testada teve associação forte (só sinais fracos com `nps_cliente`/`nota_google`, que têm nulo próprio alto) | Sem driver identificável — comportamento mais próximo de **MCAR**. | **Manter `NaN`** por ora; é o caso de menor risco para imputação por mediana caso a coluna precise ficar completa no futuro. |
| `cliente` | 15,1% | `metodo_pagamento` (Fiado/Mensal = 0% nulo; Dinheiro = 21,9% nulo) | Faz total sentido: fiado exige saber quem é o cliente para cobrar depois; pagamento em dinheiro é o mais anônimo. | **Manter `NaN`** — representa legitimamente "cliente não identificado", não erro de coleta. |
| `atendente_pagamento` | 12,9% | `metodo_pagamento` (cartão/Pix ~4-5% nulo; Dinheiro 27,8%; Fiado/Mensal 40,1%) | Pagamentos em cartão/Pix passam por máquina que já registra o atendente; dinheiro/fiado é tratado de forma mais informal. | **Manter `NaN`** — gap real de registro ligado à forma de pagamento, não dá pra adivinhar quem atendeu. |
| `metodo_pagamento` | 8,0% | Nenhuma associação significativa | **MCAR** | Sem padrão para explorar. Manter `NaN` (percentual baixo, sem urgência de imputar). |
| `funcionario_lavagem` | 6,0% | Nenhuma associação significativa | **MCAR** | Mesma decisão. |
| `tipo_carro` | 3,0% | Nenhuma associação significativa | **MCAR** | Mesma decisão. |

**Nenhuma linha será removida por causa de nulos.** Os nulos ficam concentrados em colunas específicas — apagar linhas descartaria informação boa das outras 20+ colunas por causa de uma coluna só. (A única remoção de linhas do notebook é a da Seção 5, por regra de negócio — não relacionada a nulos.)

In [ ]:
# Único tratamento aplicado: cera_ml. Todas as outras colunas permanecem NaN (ver justificativa acima).
nulos_cera_antes = df["cera_ml"].isna().sum()

df["cera_ml"] = df["cera_ml"].fillna(0)

print(f"cera_ml: {nulos_cera_antes} nulos preenchidos com 0 (assumindo 'não usou cera').")
print(f"cera_ml nulos restantes: {df['cera_ml'].isna().sum()}")

# Conferência: nenhuma outra coluna do plano foi alterada
colunas_mantidas_nan = [c for c in colunas_com_nulos if c != "cera_ml"]
nulos_ainda_presentes = df[colunas_mantidas_nan].isna().sum()
print("\nNulos mantidos como estavam (por decisão documentada acima):")
print(nulos_ainda_presentes.to_string())

## 9. Checar consistência das colunas categóricas

Um problema comum em bases reais é a mesma categoria escrita de formas diferentes (ex: "Hatch" vs "hatch " vs "HATCH"), o que faz o pandas contar como categorias distintas. Vamos olhar os valores únicos de cada coluna categórica para confirmar que isso não acontece aqui.

In [20]:
colunas_categoricas = ["tipo_carro", "dia_semana", "metodo_pagamento", "cnpj_receita"]

for coluna in colunas_categoricas:
    print(f"--- {coluna} ---")
    print(df[coluna].value_counts(dropna=False))
    print()

--- tipo_carro ---
tipo_carro
Hatch         69177
Sedã          49574
SUV           39214
Picape        23921
Utilitário     9962
NaN            5952
Name: count, dtype: int64

--- dia_semana ---
dia_semana
Sábado     47729
Domingo    40222
Sexta      34939
Quinta     25811
Quarta     20463
Segunda    14427
Terça      14209
Name: count, dtype: int64

--- metodo_pagamento ---
metodo_pagamento
Cartão de débito     59917
Cartão de crédito    53558
Dinheiro             43490
NaN                  15757
Fiado/Mensal         15598
Pix                   9480
Name: count, dtype: int64

--- cnpj_receita ---
cnpj_receita
Claudemir Nogueira Sanches ME       153240
Nog Car Estética Automotiva Ltda     44560
Name: count, dtype: int64



**Observação sobre `cnpj_receita`:** existem dois valores — "Claudemir Nogueira Sanches ME" e "Nog Car Estética Automotiva Ltda". Não é inconsistência de digitação: ao cruzar com o ano (célula abaixo), dá pra ver que o negócio migrou de um CNPJ para o outro por volta de 2016, com um período de transição em que os dois convivem. **Não deve ser unificado/corrigido** — é histórico real da empresa, não erro de dado.

In [21]:
df.groupby("ano")["cnpj_receita"].value_counts()

ano   cnpj_receita                    
2006  Claudemir Nogueira Sanches ME        5200
2007  Claudemir Nogueira Sanches ME        8600
2008  Claudemir Nogueira Sanches ME       11800
2009  Claudemir Nogueira Sanches ME       13200
2010  Claudemir Nogueira Sanches ME       13800
2011  Claudemir Nogueira Sanches ME       14100
2012  Claudemir Nogueira Sanches ME       14000
2013  Claudemir Nogueira Sanches ME       13600
2014  Claudemir Nogueira Sanches ME       13100
2015  Claudemir Nogueira Sanches ME       12400
2016  Claudemir Nogueira Sanches ME        6147
      Nog Car Estética Automotiva Ltda     5053
2017  Claudemir Nogueira Sanches ME        5573
      Nog Car Estética Automotiva Ltda     4627
2018  Claudemir Nogueira Sanches ME        5033
      Nog Car Estética Automotiva Ltda     4267
2019  Claudemir Nogueira Sanches ME        4555
      Nog Car Estética Automotiva Ltda     3645
2020  Nog Car Estética Automotiva Ltda     2810
      Claudemir Nogueira Sanches ME        2790
2

## 10. Checar faixas numéricas e outliers

Duas checagens objetivas:
- Nenhuma coluna que representa tempo, preço ou volume deveria ter valores **negativos**.
- `nps_cliente` (escala 0–10) e `nota_google` (escala 1–5) devem respeitar seus limites conhecidos.

In [22]:
colunas_devem_ser_positivas = [
    "t_teto_vidros_min", "t_capo_parabrisa_min", "t_laterais_portas_min", "t_traseira_min",
    "t_rodas_pneus_min", "t_interior_min", "tempo_lavagem_total_min",
    "tempo_espera_antes_lavagem_min", "tempo_pos_lavagem_ate_retirada_min", "tempo_ate_pagamento_min",
    "preco_reais", "shampoo_ml", "cera_ml", "pretinho_ml", "aromatizante_ml",
]

negativos = {col: int((df[col] < 0).sum()) for col in colunas_devem_ser_positivas}
negativos = {col: qtd for col, qtd in negativos.items() if qtd > 0}
print("Colunas com valores negativos (deveriam ser 0):", negativos if negativos else "nenhuma")

fora_da_escala_nps = df["nps_cliente"].dropna()
fora_da_escala_nps = fora_da_escala_nps[(fora_da_escala_nps < 0) | (fora_da_escala_nps > 10)]
print(f"nps_cliente fora da escala 0-10: {len(fora_da_escala_nps)}")

fora_da_escala_google = df["nota_google"].dropna()
fora_da_escala_google = fora_da_escala_google[(fora_da_escala_google < 0) | (fora_da_escala_google > 5)]
print(f"nota_google fora da escala 0-5: {len(fora_da_escala_google)}")

Colunas com valores negativos (deveriam ser 0): nenhuma
nps_cliente fora da escala 0-10: 0
nota_google fora da escala 0-5: 0


## 11. Exportar a base limpa

Salvamos o resultado em um **arquivo novo**, mantendo o Excel original intacto. O que mudou em relação ao original:
- removidas as **40.222 lavagens registradas aos domingos** (loja fechada nesse dia — inconsistência de regra de negócio, Seção 5);
- a coluna `data` agora é um tipo `datetime` de verdade (no Excel isso aparece como data formatada, não texto);
- `cera_ml` teve seus nulos preenchidos com `0`, com a justificativa documentada na Seção 8 (assumindo "não usou cera" para veículos que não tiveram esse produto registrado).

Todos os outros valores nulos permanecem como estavam — cada um tem uma explicação de negócio documentada (Seção 8) para não ser preenchido.

In [ ]:
ARQUIVO_LIMPO = "Base de Dados - Lava Rapido Nogueira (limpa).xlsx"

df.to_excel(ARQUIVO_LIMPO, sheet_name="lavagens", index=False)
print(f"Base limpa salva em: {ARQUIVO_LIMPO}")